<a href="https://colab.research.google.com/github/essence-git/esssencegit-workflow/blob/main/stage1_data_preparation_(2).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Stage 1 — Data Preparation
### 19-feature set · MinMaxScaler · raw baselines for explanation layer · source IP preserved · 70/15/15 split

In [1]:
import copy, gc, random, os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import (
    confusion_matrix, roc_auc_score, roc_curve,
    precision_recall_curve, classification_report, accuracy_score
)
from sklearn.tree import DecisionTreeClassifier, export_text
from torch.utils.data import DataLoader, TensorDataset

# ── Reproducibility ──────────────────────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark     = False

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device : {DEVICE}')
if DEVICE.type == 'cuda':
    print(f'GPU    : {torch.cuda.get_device_name(0)}')

Device : cpu


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 1.1 — Load CSV and verify labels

In [3]:
DATA_PATH = '/content/drive/MyDrive/sampled_cic_iot_diad.csv'

if not os.path.exists(DATA_PATH):
    raise FileNotFoundError(f"Dataset not found at '{DATA_PATH}'. Update DATA_PATH.")

raw_df = pd.read_csv(DATA_PATH, low_memory=False)
print(f'Raw shape : {raw_df.shape}')
print(f'Columns   : {len(raw_df.columns)}')
print(f'\nLabel distribution:')
print(raw_df['Label'].value_counts())

# Guard: confirm no NeedManualLabel values leaked into Label column
bad = raw_df['Label'].str.contains('NeedManualLabel', na=False).sum()
if bad > 0:
    print(f'\nWARNING: {bad} rows have NeedManualLabel in Label column.')
    print('These rows will be dropped — labels must come from folder name, not CSV column.')
    raw_df = raw_df[~raw_df['Label'].str.contains('NeedManualLabel', na=False)].reset_index(drop=True)
    print(f'Shape after drop: {raw_df.shape}')
else:
    print('\nLabel column clean — no NeedManualLabel values found.')

Raw shape : (511949, 85)
Columns   : 85

Label distribution:
Label
Benign        398330
DDoS           40000
DoS            30000
Spoofing       20000
Mirai          10000
Recon          10000
BruteForce      3619
Name: count, dtype: int64

Label column clean — no NeedManualLabel values found.


## 1.2 — Preserve source IP / port for alert explanations

Source IP, port, destination IP and port are **never fed into the model** — they are dropped before scaling.
We store them here in a separate lookup array, index-aligned with the raw dataframe, so that
Stage 7 can tell an analyst exactly where flagged traffic came from.

In [4]:
SOURCE_COLS = ['Src IP', 'Src Port', 'Dst IP', 'Dst Port']
available_source_cols = [c for c in SOURCE_COLS if c in raw_df.columns]

if available_source_cols:
    source_lookup_raw = raw_df[available_source_cols + ['Label']].copy()
    print(f'Source lookup stored: {available_source_cols}  ({len(source_lookup_raw):,} rows)')
else:
    source_lookup_raw = None
    print('WARNING: No source IP/port columns found.')
    print('Alert explanations will note traffic context as unavailable.')

Source lookup stored: ['Src IP', 'Src Port', 'Dst IP', 'Dst Port']  (511,949 rows)


## 1.3 — Chronological sort then benign / attack split

In [5]:
if 'Timestamp' in raw_df.columns:
    raw_df['Timestamp'] = pd.to_datetime(
        raw_df['Timestamp'],
        format='%d/%m/%Y %I:%M:%S %p',
        errors='coerce'
    )
    raw_df = raw_df.sort_values('Timestamp').reset_index(drop=True)
    if source_lookup_raw is not None:
        source_lookup_raw = source_lookup_raw.reindex(raw_df.index).reset_index(drop=True)
    print(f'Date range: {raw_df["Timestamp"].min()} to {raw_df["Timestamp"].max()}')
else:
    print('No Timestamp column — using existing row order (assumed chronological).')

benign_mask = raw_df['Label'] == 'Benign'
attack_mask = ~benign_mask

benign_raw_df = raw_df[benign_mask].reset_index(drop=True)
attack_raw_df = raw_df[attack_mask].reset_index(drop=True)

if source_lookup_raw is not None:
    benign_source = source_lookup_raw[benign_mask.values].reset_index(drop=True)
    attack_source = source_lookup_raw[attack_mask.values].reset_index(drop=True)
else:
    benign_source = None
    attack_source = None

print(f'\nBenign flows : {len(benign_raw_df):,}')
print(f'Attack flows : {len(attack_raw_df):,}')
print(f'\nAttack classes:')
print(attack_raw_df['Label'].value_counts())

Date range: 2022-08-02 12:06:05 to 2023-01-18 11:29:01

Benign flows : 398,330
Attack flows : 113,619

Attack classes:
Label
DDoS          40000
DoS           30000
Spoofing      20000
Recon         10000
Mirai         10000
BruteForce     3619
Name: count, dtype: int64


## 1.4 — Benign split: Train (70%) / Val (15%) / Period B (15%)

All three partitions are benign-only and in chronological order:
- **Train** — what the autoencoder learns as normal behaviour
- **Val** — used to set the 95th-percentile anomaly threshold (still Period A)
- **Period B** — genuinely unseen benign patterns used to simulate drift in Stage 5

In [6]:
n         = len(benign_raw_df)
train_end = int(0.70 * n)
val_end   = int(0.85 * n)

train_raw_df = benign_raw_df.iloc[:train_end].copy().reset_index(drop=True)
val_raw_df   = benign_raw_df.iloc[train_end:val_end].copy().reset_index(drop=True)
period_b_raw = benign_raw_df.iloc[val_end:].copy().reset_index(drop=True)

if benign_source is not None:
    train_source  = benign_source.iloc[:train_end].reset_index(drop=True)
    val_source    = benign_source.iloc[train_end:val_end].reset_index(drop=True)
    period_b_src  = benign_source.iloc[val_end:].reset_index(drop=True)

print(f'Train  (benign only) : {len(train_raw_df):,} rows  ({len(train_raw_df)/n:.0%})')
print(f'Val    (benign only) : {len(val_raw_df):,} rows  ({len(val_raw_df)/n:.0%})')
print(f'Period B (unseen)    : {len(period_b_raw):,} rows  ({len(period_b_raw)/n:.0%})')

Train  (benign only) : 278,831 rows  (70%)
Val    (benign only) : 59,749 rows  (15%)
Period B (unseen)    : 59,750 rows  (15%)


## 1.5 — Test set: Period B benign + stratified attack sample

Up to 10,000 flows per attack class. BruteForce is sparse (~3,619 rows in full CIC dataset)
so all available rows are used — noted as a dataset limitation.

In [7]:
ATTACK_SAMPLE_PER_CLASS = 10_000

attack_frames        = []
attack_source_frames = []

for atype in attack_raw_df['Label'].unique():
    mask   = attack_raw_df['Label'] == atype
    subset = attack_raw_df[mask].reset_index(drop=True)
    n_take = min(len(subset), ATTACK_SAMPLE_PER_CLASS)
    idx    = subset.sample(n=n_take, random_state=SEED).index
    attack_frames.append(subset.loc[idx])
    if attack_source is not None:
        src_sub = attack_source[mask.values].reset_index(drop=True)
        attack_source_frames.append(src_sub.loc[idx])
    note = ' <- sparse (dataset limitation)' if n_take < ATTACK_SAMPLE_PER_CLASS else ''
    print(f'  {atype:<15}: {n_take:,}{note}')

test_attack_raw = pd.concat(attack_frames, ignore_index=True)
test_raw_df     = pd.concat([period_b_raw, test_attack_raw], ignore_index=True)

test_labels_binary = pd.Series(
    [0] * len(period_b_raw) + [1] * len(test_attack_raw), name='is_attack'
)
test_class_labels = pd.concat([
    pd.Series(['Benign'] * len(period_b_raw)),
    test_attack_raw['Label'].reset_index(drop=True)
], ignore_index=True)

if benign_source is not None and attack_source_frames:
    test_attack_src = pd.concat(attack_source_frames, ignore_index=True)
    test_source     = pd.concat([period_b_src, test_attack_src], ignore_index=True)
else:
    test_source = None

print(f'\nTest set total      : {len(test_raw_df):,}')
print(f'  Benign (Period B) : {len(period_b_raw):,}')
print(f'  Attack (sampled)  : {len(test_attack_raw):,}')
print(f'  Attack proportion : {test_labels_binary.mean():.1%}')
gc.collect()

  DoS            : 10,000
  DDoS           : 10,000
  Recon          : 10,000
  BruteForce     : 3,619 <- sparse (dataset limitation)
  Spoofing       : 10,000
  Mirai          : 10,000

Test set total      : 113,369
  Benign (Period B) : 59,750
  Attack (sampled)  : 53,619
  Attack proportion : 47.3%


53

## 1.6 — Feature engineering: unit fix · 19-feature rename · clean

These are the exact 19 features from the original notebook, kept as-is.
`Flow Duration` is divided by 1000 (microseconds → seconds) before renaming.
NaN/inf replaced, 99th-percentile caps computed on training data only.

In [8]:
COLS_TO_DROP = [
    'Flow ID', 'Src IP', 'Src Port', 'Dst IP', 'Dst Port',
    'Timestamp', 'Label', 'Subtype_Label'
]

# ── 19 features — original notebook names, kept exactly ──────────────────────
RENAME_MAP = {
    'Flow Duration'              : 'FLOW_DURATION_SECONDS',
    'Total Length of Fwd Packet' : 'IN_BYTES',
    'Total Length of Bwd Packet' : 'OUT_BYTES',
    'Protocol'                   : 'PROTOCOL',
    'Packet Length Min'          : 'MIN_IP_PKT_LEN',
    'Fwd Packet Length Max'      : 'LONGEST_FLOW_PKT',
    'Fwd Packet Length Min'      : 'SHORTEST_FLOW_PKT',
    'Flow Bytes/s'               : 'SRC_TO_DST_SECOND_BYTES',
    'Fwd IAT Mean'               : 'SRC_TO_DST_IAT_AVG',
    'Fwd IAT Min'                : 'SRC_TO_DST_IAT_MIN',
    'FWD Init Win Bytes'         : 'TCP_WIN_MAX_IN',
    'Bwd Init Win Bytes'         : 'TCP_WIN_MAX_OUT',
    'SYN Flag Count'             : 'SYN_FLAG_CNT',
    'RST Flag Count'             : 'RST_FLAG_CNT',
    'FIN Flag Count'             : 'FIN_FLAG_CNT',
    'Down/Up Ratio'              : 'DOWN_UP_RATIO',
    'Packet Length Mean'         : 'PKT_LEN_MEAN',
    'Flow IAT Mean'              : 'FLOW_IAT_MEAN',
    'Active Mean'                : 'ACTIVE_MEAN',
}

FEATURES = list(RENAME_MAP.values())  # 19 features
print(f'Feature count : {len(FEATURES)}')
print(f'Features      : {FEATURES}')

# Guard: confirm all 19 source columns exist in the CSV
missing = [k for k in RENAME_MAP if k not in raw_df.columns]
if missing:
    raise ValueError(
        f'Cannot proceed — {len(missing)} column(s) not found in CSV:\n'
        + '\n'.join(f'  {c}' for c in missing)
    )
print('\nAll 19 source columns confirmed present in CSV.')

Feature count : 19
Features      : ['FLOW_DURATION_SECONDS', 'IN_BYTES', 'OUT_BYTES', 'PROTOCOL', 'MIN_IP_PKT_LEN', 'LONGEST_FLOW_PKT', 'SHORTEST_FLOW_PKT', 'SRC_TO_DST_SECOND_BYTES', 'SRC_TO_DST_IAT_AVG', 'SRC_TO_DST_IAT_MIN', 'TCP_WIN_MAX_IN', 'TCP_WIN_MAX_OUT', 'SYN_FLAG_CNT', 'RST_FLAG_CNT', 'FIN_FLAG_CNT', 'DOWN_UP_RATIO', 'PKT_LEN_MEAN', 'FLOW_IAT_MEAN', 'ACTIVE_MEAN']

All 19 source columns confirmed present in CSV.


In [9]:
def transform_pipeline(df, is_train=False, train_means=None, caps=None):
    """
    Feature engineering pipeline applied identically to every split.

    Steps:
      1. Drop identifier columns
      2. Unit fix: Flow Duration microseconds -> seconds (divide by 1000)
      3. Rename to internal feature names
      4. Keep only the 19 aligned features
      5. Replace inf/-inf with NaN
      6. Fill NaN with training means (no data leakage on val/test)
      7. Clip at 99th-percentile caps (computed from training set only)

    Returns
    -------
    is_train=True  : (processed_df, train_means, caps)
    is_train=False : processed_df
    """
    out = df.drop(columns=COLS_TO_DROP, errors='ignore').copy()

    if 'Flow Duration' in out.columns:
        out['Flow Duration'] = out['Flow Duration'] / 1000.0

    out = out.rename(columns=RENAME_MAP)
    out = out[FEATURES]
    out = out.replace([np.inf, -np.inf], np.nan)

    if is_train:
        computed_means = out.mean(numeric_only=True)
        out = out.fillna(computed_means)
        computed_caps  = {col: out[col].quantile(0.99) for col in out.columns}
        for col, cap in computed_caps.items():
            out[col] = out[col].clip(upper=cap)
        return out, computed_means, computed_caps
    else:
        out = out.fillna(train_means)
        for col, cap in caps.items():
            out[col] = out[col].clip(upper=cap)
        return out


train_df, train_means_raw, caps = transform_pipeline(train_raw_df, is_train=True)
val_df      = transform_pipeline(val_raw_df,   is_train=False, train_means=train_means_raw, caps=caps)
period_b_df = transform_pipeline(period_b_raw, is_train=False, train_means=train_means_raw, caps=caps)
test_df     = transform_pipeline(test_raw_df,  is_train=False, train_means=train_means_raw, caps=caps)

print('Feature engineering complete.')
print(f'  train_df    : {train_df.shape}')
print(f'  val_df      : {val_df.shape}')
print(f'  period_b_df : {period_b_df.shape}')
print(f'  test_df     : {test_df.shape}')

Feature engineering complete.
  train_df    : (278831, 19)
  val_df      : (59749, 19)
  period_b_df : (59750, 19)
  test_df     : (113369, 19)


## 1.7 — Store raw baseline means for the explanation layer

The explanation layer in Stage 7 computes:

```
ratio = alert_window_mean[feature] / baseline_means_raw[feature]
```

and renders this as *"SYN_FLAG_CNT was 847.00 in the last 10 flows, which is 34.2x above its normal baseline of 24.77"*.

This requires the denominator to be in **original unscaled units**.
`baseline_means_raw` is stored here from the post-transform, pre-scale training data.

In [10]:
baseline_means_raw = train_means_raw.copy()

print('Baseline means — original units (explanation layer denominator):')
print('-' * 52)
for feat, val in baseline_means_raw.items():
    print(f'  {feat:<30} {val:>14.4f}')

Baseline means — original units (explanation layer denominator):
----------------------------------------------------
  FLOW_DURATION_SECONDS              24485.5039
  IN_BYTES                           11482.4961
  OUT_BYTES                           2001.1176
  PROTOCOL                              12.6376
  MIN_IP_PKT_LEN                        39.7587
  LONGEST_FLOW_PKT                     214.1444
  SHORTEST_FLOW_PKT                     44.5044
  SRC_TO_DST_SECOND_BYTES           932900.3897
  SRC_TO_DST_IAT_AVG               4530519.0581
  SRC_TO_DST_IAT_MIN               2125212.9568
  TCP_WIN_MAX_IN                      4145.4116
  TCP_WIN_MAX_OUT                      862.4988
  SYN_FLAG_CNT                           0.4241
  RST_FLAG_CNT                           0.0453
  FIN_FLAG_CNT                           0.3450
  DOWN_UP_RATIO                          0.6007
  PKT_LEN_MEAN                          88.4538
  FLOW_IAT_MEAN                    3161146.3594
  ACTIVE_MEAN     

## 1.8 — MinMaxScaler (fit on training set only)

MinMaxScaler chosen over StandardScaler because:
- IoT features (packet counts, byte counts, flag counters) are heavily right-skewed;
  StandardScaler's Gaussian assumption does not hold for this data
- Compresses all values to [0, 1] while preserving relative magnitude
- Exact inverse transform makes it straightforward to recover original units
  for the explanation layer

Fitted **once** on training data. The same fitted scaler is reused on every other split.

In [11]:
scaler = MinMaxScaler()
train_scaled    = scaler.fit_transform(train_df)
val_scaled      = scaler.transform(val_df)
period_b_scaled = scaler.transform(period_b_df)
test_scaled     = scaler.transform(test_df)

print('MinMaxScaler fitted on training set.')
print(f'  train_scaled    : {train_scaled.shape}   min={train_scaled.min():.3f}  max={train_scaled.max():.3f}')
print(f'  val_scaled      : {val_scaled.shape}')
print(f'  period_b_scaled : {period_b_scaled.shape}')
print(f'  test_scaled     : {test_scaled.shape}')

MinMaxScaler fitted on training set.
  train_scaled    : (278831, 19)   min=0.000  max=1.000
  val_scaled      : (59749, 19)
  period_b_scaled : (59750, 19)
  test_scaled     : (113369, 19)


## 1.9 — Sliding window sequences (window=10, step=1)

Each sequence is a (10, 19) array — 10 consecutive flows, 19 features.
Label of a sequence = label of its **last flow** (index `i + window - 1`).
Stride-trick builder avoids copying data.

In [12]:
WINDOW = 10

def build_sequences(data, window=WINDOW):
    """Sliding window using stride tricks — no data copied."""
    n, f    = data.shape
    shape   = (n - window + 1, window, f)
    strides = (data.strides[0], data.strides[0], data.strides[1])
    return np.lib.stride_tricks.as_strided(data, shape=shape, strides=strides)

X_train    = build_sequences(train_scaled).copy()     # .copy() makes array writable for PyTorch
X_val      = build_sequences(val_scaled).copy()
X_period_b = build_sequences(period_b_scaled).copy()
X_test     = build_sequences(test_scaled).copy()

# Binary ground-truth labels for test sequences
y_test = np.array([
    test_labels_binary.iloc[i + WINDOW - 1]
    for i in range(len(test_scaled) - WINDOW + 1)
])

# Per-class labels for the Stage 3 breakdown table
y_test_class = np.array([
    test_class_labels.iloc[i + WINDOW - 1]
    for i in range(len(test_scaled) - WINDOW + 1)
])

print(f'X_train    : {X_train.shape}   (sequences x window x features)')
print(f'X_val      : {X_val.shape}')
print(f'X_period_b : {X_period_b.shape}  <- drift stream (Stage 5)')
print(f'X_test     : {X_test.shape}')
print(f'y_test     : {y_test.shape}   attack rate = {y_test.mean():.3f}')
print(f'\nTest class breakdown:')
for cls, cnt in pd.Series(y_test_class).value_counts().items():
    print(f'  {cls:<15}: {cnt:,}')

gc.collect()

X_train    : (278822, 10, 19)   (sequences x window x features)
X_val      : (59740, 10, 19)
X_period_b : (59741, 10, 19)  <- drift stream (Stage 5)
X_test     : (113360, 10, 19)
y_test     : (113360,)   attack rate = 0.473

Test class breakdown:
  Benign         : 59,741
  DoS            : 10,000
  DDoS           : 10,000
  Recon          : 10,000
  Spoofing       : 10,000
  Mirai          : 10,000
  BruteForce     : 3,619


0

## 1.10 — Source context lookup for alert explanations

`seq_to_source(seq_idx, source_df)` retrieves the source/destination IP and port
of the triggering flow for any flagged sequence. Used by the explanation layer in Stage 7.

In [13]:
def seq_to_source(seq_idx, source_df, window=WINDOW):
    """
    Returns traffic source context for the last flow in a flagged sequence.

    Parameters
    ----------
    seq_idx   : index into the sequence array (e.g. X_test)
    source_df : source lookup DataFrame aligned with the same raw split
    window    : sequence window size

    Returns
    -------
    dict — keys: Src IP, Src Port, Dst IP, Dst Port (where available)
    """
    if source_df is None:
        return {'note': 'Source IP/port not available in this dataset'}
    row_idx = min(seq_idx + window - 1, len(source_df) - 1)
    return source_df.iloc[row_idx].to_dict()


if test_source is not None:
    sample = seq_to_source(0, test_source)
    print('Source context smoke test (sequence 0):')
    for k, v in sample.items():
        print(f'  {k:<12}: {v}')
else:
    print('Source lookup unavailable — explanation layer will note this.')

Source context smoke test (sequence 0):
  Src IP      : 192.168.137.12
  Src Port    : 46635
  Dst IP      : 192.168.137.91
  Dst Port    : 1443
  Label       : DDoS


## 1.11 — Stage 1 handoff summary

In [14]:
print('=' * 60)
print('STAGE 1 COMPLETE')
print('=' * 60)
print(f'Features          : {len(FEATURES)} (19-feature set)')
print(f'Scaler            : MinMaxScaler  range [0, 1]')
print(f'Window            : {WINDOW} flows')
print()
print('Sequence arrays:')
print(f'  X_train    : {X_train.shape}')
print(f'  X_val      : {X_val.shape}')
print(f'  X_period_b : {X_period_b.shape}   drift stream Stage 5')
print(f'  X_test     : {X_test.shape}')
print(f'  y_test     : {y_test.shape}')
print()
print('Explanation layer assets:')
print(f'  baseline_means_raw : {len(baseline_means_raw)} features in original units')
print(f'  scaler             : MinMaxScaler fitted')
print(f'  test_source        : {"available (" + str(len(test_source)) + " rows)" if test_source is not None else "unavailable"}')
print(f'  seq_to_source()    : ready')
print('=' * 60)

STAGE 1 COMPLETE
Features          : 19 (19-feature set)
Scaler            : MinMaxScaler  range [0, 1]
Window            : 10 flows

Sequence arrays:
  X_train    : (278822, 10, 19)
  X_val      : (59740, 10, 19)
  X_period_b : (59741, 10, 19)   drift stream Stage 5
  X_test     : (113360, 10, 19)
  y_test     : (113360,)

Explanation layer assets:
  baseline_means_raw : 19 features in original units
  scaler             : MinMaxScaler fitted
  test_source        : available (113369 rows)
  seq_to_source()    : ready


In [15]:
import pickle, numpy as np

np.save('X_train.npy', X_train)
np.save('X_val.npy', X_val)
np.save('X_period_b.npy', X_period_b)
np.save('X_test.npy', X_test)
np.save('y_test.npy', y_test)
np.save('y_test_class.npy', y_test_class)

with open('stage1_assets.pkl', 'wb') as f:
    pickle.dump({
        'baseline_means_raw': baseline_means_raw,
        'scaler':             scaler,
        'caps':               caps,
        'train_means_raw':    train_means_raw,
        'FEATURES':           FEATURES,
        'WINDOW':             WINDOW,
        'test_source':        test_source,
    }, f)

print('Stage 1 assets saved.')

Stage 1 assets saved.


In [16]:
from google.colab import drive
drive.mount('/content/drive')

import pickle, numpy as np

SAVE_DIR = '/content/drive/MyDrive/gru_iot_project/'
import os; os.makedirs(SAVE_DIR, exist_ok=True)

np.save(SAVE_DIR + 'X_train.npy',    X_train)
np.save(SAVE_DIR + 'X_val.npy',      X_val)
np.save(SAVE_DIR + 'X_period_b.npy', X_period_b)
np.save(SAVE_DIR + 'X_test.npy',     X_test)
np.save(SAVE_DIR + 'y_test.npy',     y_test)
np.save(SAVE_DIR + 'y_test_class.npy', y_test_class)

with open(SAVE_DIR + 'stage1_assets.pkl', 'wb') as f:
    pickle.dump({
        'baseline_means_raw': baseline_means_raw,
        'scaler':             scaler,
        'caps':               caps,
        'train_means_raw':    train_means_raw,
        'FEATURES':           FEATURES,
        'WINDOW':             WINDOW,
        'test_source':        test_source,
    }, f)

print('Stage 1 assets saved to Google Drive.')
print(f'Files in {SAVE_DIR}:')
for f in os.listdir(SAVE_DIR):
    size = os.path.getsize(SAVE_DIR + f) / 1024 / 1024
    print(f'  {f:<30} {size:.1f} MB')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Stage 1 assets saved to Google Drive.
Files in /content/drive/MyDrive/gru_iot_project/:
  X_train.npy                    404.2 MB
  X_val.npy                      86.6 MB
  X_period_b.npy                 86.6 MB
  X_test.npy                     164.3 MB
  y_test.npy                     0.9 MB
  y_test_class.npy               4.3 MB
  stage1_assets.pkl              3.4 MB
